# Saga-Style Compensation
Companion notebook for the **Saga-Style Compensation** section of Chapter 8.

When a later step fails **after** earlier steps already changed real systems, simple rollback isn't enough -- some actions (notifications, external calls) can't be fully undone. Instead, register a compensation for each step as soon as it succeeds, and run those compensations in reverse order if a later step fails.

The book's own vendor-onboarding workflow state, as a real Python object:

In [1]:
workflow = {
    "run_id": "onboard-842",
    "steps": [
        {"name": "create_vendor", "status": "pending", "compensate": "mark_pending_review"},
        {"name": "reserve_budget", "status": "pending", "compensate": "release_budget"},
        {"name": "send_approval", "status": "pending", "compensate": "send_correction"},
        {"name": "create_payment", "status": "pending", "compensate": None},
    ],
}

A small saga runner: forward actions and their matching compensations, keyed by step name, run against `workflow` above.

In [2]:
def create_vendor():
    return {"vendor_id": "V-9001"}


def reserve_budget():
    return {"reservation_id": "RSV-1"}


def send_approval():
    return {"approval_id": "APR-1"}


def create_payment():
    raise RuntimeError("payment processor timeout")


FORWARD_ACTIONS = {
    "create_vendor": create_vendor,
    "reserve_budget": reserve_budget,
    "send_approval": send_approval,
    "create_payment": create_payment,
}

COMPENSATIONS = {
    "mark_pending_review": lambda: print("  compensating: marking vendor as pending_review"),
    "release_budget": lambda: print("  compensating: releasing reserved budget"),
    "send_correction": lambda: print("  compensating: sending correction message for the approval request"),
}

In [3]:
def run_saga(workflow: dict) -> dict:
    completed_compensations = []  # stack: registered as soon as a forward step succeeds

    for step in workflow["steps"]:
        try:
            FORWARD_ACTIONS[step["name"]]()
            step["status"] = "done"
            if step["compensate"]:
                completed_compensations.append(step["compensate"])
        except Exception as e:
            step["status"] = "failed"
            print(f"step {step['name']!r} failed: {e}")
            print("running compensations in reverse order:")
            for compensation_name in reversed(completed_compensations):
                COMPENSATIONS[compensation_name]()
            workflow["status"] = "compensated"
            return workflow

    workflow["status"] = "completed"
    return workflow


final_workflow = run_saga(workflow)
print()
print("final workflow state:")
for step in final_workflow["steps"]:
    print(f"  {step['name']}: {step['status']}")
print("overall:", final_workflow["status"])

step 'create_payment' failed: payment processor timeout
running compensations in reverse order:
  compensating: sending correction message for the approval request
  compensating: releasing reserved budget
  compensating: marking vendor as pending_review

final workflow state:
  create_vendor: done
  reserve_budget: done
  send_approval: done
  create_payment: failed
overall: compensated


Compensation is not always an "undo" -- `mark_pending_review` keeps the vendor record instead of deleting it, restoring business consistency rather than pretending the onboarding never started.

If a compensation itself fails, don't hide it -- mark the run for manual recovery with the full audit trail, rather than silently continuing:

In [4]:
def run_saga_safe(workflow: dict) -> dict:
    completed_compensations = []
    audit_trail = []

    for step in workflow["steps"]:
        try:
            FORWARD_ACTIONS[step["name"]]()
            step["status"] = "done"
            audit_trail.append(f"{step['name']}: succeeded")
            if step["compensate"]:
                completed_compensations.append(step["compensate"])
        except Exception as e:
            step["status"] = "failed"
            audit_trail.append(f"{step['name']}: failed ({e})")
            for compensation_name in reversed(completed_compensations):
                try:
                    COMPENSATIONS[compensation_name]()
                    audit_trail.append(f"compensation {compensation_name}: succeeded")
                except Exception as comp_error:
                    audit_trail.append(f"compensation {compensation_name}: FAILED ({comp_error})")
                    workflow["status"] = "needs_manual_recovery"
                    workflow["audit_trail"] = audit_trail
                    return workflow
            workflow["status"] = "compensated"
            workflow["audit_trail"] = audit_trail
            return workflow

    workflow["status"] = "completed"
    workflow["audit_trail"] = audit_trail
    return workflow